# EDA on PCAP derived Files

This separate EDA is necessary to completely understand the new dataset that we derived using raw PCAPs.

## Imports & Initial Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from src.core.config import config_loader

In [2]:
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option('display.max_columns', None)

schema_cfg = config_loader('../config/preprocessing/validation_schema.yaml')
non_numeric_cols = schema_cfg['non_numeric_col']
report_file_path = Path('../reports/pcap_derived.txt')
report_file_path.parent.mkdir(parents=True, exist_ok=True)

## Loading Dataset

In [3]:
dataset_dir = Path('../dataset/extracted/labeled')
dataset_files = os.listdir(dataset_dir)

## Function Definitions

Here are some functions defined to be used later in this notebook.

In [4]:
def drop_duplicate_header(df):
    header = df.columns

    header_rows = (df.astype(str) == header).all(axis=1)

    df = df.loc[~header_rows].copy()

    feature_cols = df.columns.drop(non_numeric_cols)

    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    return df

In [5]:
def generate_correlation_report(df, output_path, threshold=0.90, top_n=10):
    if list(df.columns) == list(df.index) and df.shape[0] == df.shape[1]:
        corr = df
    else:
        corr = df.corr(numeric_only=True)

    cols = corr.columns.tolist()
    n = len(cols)
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            v = corr.iloc[i, j]
            if pd.isna(v):
                continue
            pairs.append((cols[i], cols[j], float(v)))

    categories = [
        ("Perfect Positive Correlation (r = +1.000)", lambda r: r >= 0.9999995, True),
        ("Very High Positive Correlation (0.95 <= r < 1.00)", lambda r: 0.95 <= r < 0.9999995, True),
        (f"High Positive Correlation ({threshold:.2f} <= r < 0.95)", lambda r: threshold <= r < 0.95, True),
        (f"High Negative Correlation (r <= -{threshold:.2f})", lambda r: r <= -threshold, False),
    ]

    with open(output_path, "a") as f:
        for title, cond, descending in categories:
            matched = [p for p in pairs if cond(p[2])]
            if matched:
                selected = sorted(matched, key=lambda p: p[2], reverse=descending)
            else:
                selected = sorted(pairs, key=lambda p: p[2], reverse=descending)[:top_n]

            f.write(f"\n{title}\n")
            for a, b, v in selected:
                f.write(f"{a} and {b} correlate by {v:+.6f}\n")

In [6]:
for dataset_file in dataset_files:
    i = 1
    with open(report_file_path, 'a') as f:
        f.write(f"Reports for file: {dataset_file}")
    for chunk in pd.read_csv(Path(dataset_dir, dataset_file), chunksize=3_000_000, low_memory=False):
        chunk = drop_duplicate_header(chunk)

        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.dropna(inplace=True)

        negative_counts = (chunk.select_dtypes(include="number") < 0).sum()

        with open(report_file_path, 'a') as f:
            f.write(f'Negative Count Report #{i}\n for file: {dataset_file}')
            f.write(negative_counts[negative_counts > 0].to_string())
            f.write('\n')
            for col in chunk.columns:
                if chunk[col].dtype in ['int64', 'float64']:
                    f.write(f"{col} {chunk[chunk[col] < 0][col].unique()}\n")
            f.write('\n')

            f.write(f'Constant Feature Report #{i} for file: {dataset_file}')

            for col in chunk.columns:
                if chunk[col].dtype in ['int64', 'float64'] and chunk[col].var() == 0:
                     f.write(f"{col} {chunk[col].unique()}\n")

            f.write(f'Correlation Report #{i} for file: {dataset_file}')

        generate_correlation_report(chunk, report_file_path)

        i += 1
